In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../data/dataset.csv', index_col=0)
df = df.dropna(subset=['artists'])
df = df.drop_duplicates(subset=['track_id'])
df.head()

## Serie de tiempo: popularidad promedio por género

El dataset no tiene fechas, así que construimos una serie de tiempo sintética.
Ordenamos los géneros por popularidad promedio y los tratamos como puntos en el tiempo.

In [ ]:
serie = df.groupby('track_genre')['popularity'].mean().sort_values().reset_index()
serie.columns = ['genre', 'avg_popularity']
y = serie['avg_popularity'].values
T = len(y)
t = np.arange(T)

plt.figure(figsize=(12, 4))
plt.plot(t, y)
plt.xticks(t, serie['genre'], rotation=90, fontsize=7)
plt.ylabel('Popularidad promedio')
plt.title('Popularidad promedio por género (ordenada)')
plt.tight_layout()
plt.show()

## Análisis espectral

In [ ]:
Y = np.fft.fft(y)
freqs = np.fft.fftfreq(T)
amplitude = np.abs(Y) / T

half = T // 2
plt.figure(figsize=(10, 4))
plt.plot(freqs[:half], amplitude[:half])
plt.xlabel('Frecuencia')
plt.ylabel('Amplitud')
plt.title('Espectro de amplitud — popularidad por género')
plt.tight_layout()
plt.show()

In [ ]:
n_components = 5
idx = np.argsort(amplitude[:half])[::-1][1:n_components+1]
dominant_freqs = freqs[idx]
dominant_amps = amplitude[idx]
print('Frecuencias dominantes:', dominant_freqs)
print('Amplitudes:', dominant_amps)

In [ ]:
trend = np.polyval(np.polyfit(t, y, 1), t)

y_reconstructed = trend.copy()
for f, A in zip(dominant_freqs, dominant_amps):
    y_reconstructed += A * np.cos(2 * np.pi * f * t)

plt.figure(figsize=(12, 5))
plt.plot(t, y, label='Original', alpha=0.7)
plt.plot(t, y_reconstructed, label='Reconstruida', linestyle='--')
plt.plot(t, trend, label='Tendencia', linestyle=':')
plt.xticks(t[::10], serie['genre'].iloc[::10], rotation=90, fontsize=7)
plt.ylabel('Popularidad promedio')
plt.title('Serie original vs reconstruida')
plt.legend()
plt.tight_layout()
plt.show()